# AI Agent Integration Test

This notebook allows you to test the end-to-end integration of the Financial AI Agent Orchestrator, the Fundamental Analysis Agent, and the Technical Analysis Agent.

**Instructions:**
1.  Run the cells in order from top to bottom (`Runtime` > `Run all`).
2.  The first code cell will clone the required repositories and install all dependencies.
3.  The second code cell will start all three services in the background.
4.  In the final cell, enter the stock ticker you want to analyze (e.g., 'AAPL') and run it to see the result.

In [ ]:
# Clone the external agent repositories
print("Cloning external agent repositories...")
!git clone https://github.com/athipan1/Technical_Agent.git
!git clone https://github.com/athipan1/Fundamental_Agent.git
print("Repositories cloned.")

# Install dependencies for all services
print("\nInstalling dependencies...")
# The orchestrator files are in the root directory
!pip install -q -r ./requirements.txt
!pip install -q -r Technical_Agent/requirements.txt
!pip install -q -r Fundamental_Agent/requirements.txt
print("✅ Setup complete! Dependencies installed.")

In [ ]:
import os
import time

print("Starting all AI agent services in the background...")

# Start the Technical Agent
os.chdir('Technical_Agent')
!uvicorn app.main:app --host 0.0.0.0 --port 8000 > technical_agent.log 2>&1 &
os.chdir('..')
print("Technical Agent starting on port 8000.")

# Start the Fundamental Agent
os.chdir('Fundamental_Agent')
!uvicorn app.main:app --host 0.0.0.0 --port 8001 > fundamental_agent.log 2>&1 &
os.chdir('..')
print("Fundamental Agent starting on port 8001.")

# Start the Orchestrator
# The orchestrator needs to know the agent URLs
os.environ['TECHNICAL_AGENT_URL'] = 'http://localhost:8000/analyze'
os.environ['FUNDAMENTAL_AGENT_URL'] = 'http://localhost:8001/analyze'
!uvicorn app.main:app --host 0.0.0.0 --port 8080 > orchestrator.log 2>&1 &
print("Orchestrator starting on port 8080.")

print("\nWaiting 10 seconds for services to initialize...")
time.sleep(10)

print("✅ All services should be running.")

In [ ]:
import requests
import json

# --- USER INPUT ---
ticker = "AAPL" # @param {type:"string"}
# ------------------

print(f"Sending request to Orchestrator for ticker: {ticker}...")

orchestrator_url = "http://localhost:8080/analyze"
request_body = {"ticker": ticker}

try:
    response = requests.post(orchestrator_url, json=request_body, timeout=20)
    response.raise_for_status()  # Raise an exception for bad status codes

    # Pretty-print the JSON response
    response_data = response.json()
    print("\n--- Orchestrator Response ---")
    print(json.dumps(response_data, indent=2))
    print("--------------------------")

except requests.exceptions.RequestException as e:
    print(f"\n❌ An error occurred: {e}")
    print("\n--- Server Logs (last 10 lines) ---")
    print("\nOrchestrator Log:")
    !tail -n 10 orchestrator.log
    print("\nTechnical Agent Log:")
    !tail -n 10 Technical_Agent/technical_agent.log
    print("\nFundamental Agent Log:")
    !tail -n 10 Fundamental_Agent/fundamental_agent.log
    print("-------------------------------------")